# 01 — Veri Toplama (Google Earth Engine)

Bu notebook projenin bütün ham verisini üretir. Sonraki notebook'lar (02–06) **Earth Engine'e hiç bağlanmadan** buradaki CSV dosyalarıyla çalışır.

| Çıktı | Ne içerir | Hangi fazda kullanılır |
|---|---|---|
| `data/processed/samples_2021.csv` | 7 şehirden etiketli örnek noktalar (bantlar + indeksler + sınıf) | 02, 03, 06 |
| `data/processed/iklim_sehir_yillik.csv` | Şehir × yıl (2003–2024): MODIS LST, ERA5-Land toprak nemi ve sıcaklık, CHIRPS yağış | 04 |
| `data/processed/grid_1km_2014_2024.csv` | 1 km hücreler: 2014 ve 2024 indeksleri, Landsat LST, MODIS LST, rakım | 04 |
| `results/tables/worldcover_dislanan_siniflar.csv` | Kar-buz / mangrov / yosun-liken alan oranı | Sınırlılıklar |
| `results/tables/su_alani_karsilastirma.csv` | JRC ve MNDWI su alanları, 2021 tutarlılığı | 04, Sınırlılıklar |
| `results/tables/lst_eksik_orani.csv` | Landsat yüzey sıcaklığında eksik piksel oranı | Sınırlılıklar |
| `results/figures/rgb/*.png` | Şehir başına 2014 ve 2024 gerçek renkli görüntü | 02 |

**Önbellek (cache) mantığı:** Uzun hesaplamalar Drive'a CSV olarak kaydedilir. Notebook tekrar çalıştırıldığında dosya zaten varsa hesaplama atlanır. Böylece Earth Engine kotası boşa harcanmaz. Her şeyi baştan hesaplatmak için `FORCE_RECOMPUTE = True` yapın.

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) özel GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Özel repoyu okumak için Colab'ın **Gizli anahtarlar (Secrets)** bölümündeki `GITHUB_TOKEN` kullanılır. Token ekrana yazdırılmaz.

In [ ]:
!pip install -q earthengine-api geemap

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git hatası: " + (r.stderr or r.stdout).replace(token, "***"))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        raise RuntimeError("Colab'da sol menüdeki anahtar simgesinden GITHUB_TOKEN ekleyin ve "
                           "'Not defteri erişimi'ni açın (README'ye bakın).") from e
    if os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["-C", CODE_DIR, "pull", "-q"], token)
    else:
        _git(["clone", "-q", f"https://{token}@github.com/{GITHUB_REPO}.git", CODE_DIR], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import ee
from src import gee_utils as gu
importlib.reload(gu)
gu.init_ee()

In [ ]:
import numpy as np
import pandas as pd

FORCE_RECOMPUTE = False
CITY_LIST = list(config.CITIES)
C = config.COLLECTIONS
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

geoms = {c: gu.city_geometry(c) for c in CITY_LIST}
cities_fc = gu.get_city_boundaries()

def with_lonlat(f):
    xy = f.geometry().coordinates()
    return f.set({"lon": xy.get(0), "lat": xy.get(1)})

def set_props(props):
    return lambda f: f.set(props)

## 1. Temel kavramlar

**Landsat 8/9 Collection 2 Level-2:** NASA/USGS uyduları. Her 16 günde bir aynı yeri 30 m çözünürlükte görüntüler (iki uydu birlikte 8 günde bir). "Level-2" atmosfer etkisi düzeltilmiş **yüzey yansıması** ve **yüzey sıcaklığı** demektir.

| Bant | Dalga boyu | Ne işe yarar |
|---|---|---|
| SR_B2 | Mavi | Su, atmosfer |
| SR_B3 | Yeşil | Bitki, su |
| SR_B4 | Kırmızı | Bitki klorofili kırmızıyı emer |
| SR_B5 | Yakın kızılötesi (NIR) | Sağlıklı bitki NIR'ı çok yansıtır |
| SR_B6 | Kısa dalga kızılötesi 1 (SWIR1) | Nem, toprak, yapı |
| SR_B7 | Kısa dalga kızılötesi 2 (SWIR2) | Toprak, mineral |
| ST_B10 | Termal | Yüzey sıcaklığı (LST) |

**Ölçek faktörü:** Landsat değerleri tamsayı olarak saklanır. Gerçek yansıma için `değer × 0.0000275 − 0.2`, sıcaklık (Kelvin) için `değer × 0.00341802 + 149` uygulanır. Sıcaklığı °C'ye çevirmek için sonuçtan 273.15 çıkarırız.

**Bulut maskesi:** Her Landsat görüntüsünde `QA_PIXEL` adlı bir kalite bandı vardır. Bu bant her piksel için "bulut mu, bulut gölgesi mi, sirrus mu" bilgisini bitler halinde tutar. Bulutlu pikselleri **maskeleriz**, yani "yok" sayarız. Aksi halde bulut beyaz göründüğü için yanlışlıkla "çıplak alan" ya da "yapı" sanılabilir.

**Kompozit (medyan):** Bir yaz boyunca (Haziran–Ağustos) aynı yerin birkaç görüntüsü olur. Bulutlu kısımları attıktan sonra her piksel için kalan değerlerin **medyanını** alırız. Sonuç tek ve bulutsuz bir görüntüdür. Medyan, ortalamaya göre kalan bulut kenarlarına ve gölgelere daha dayanıklıdır. Hep yazı seçmemizin nedeni, yıllar arasında mevsim farkını ortadan kaldırmak ve bitki örtüsünün en belirgin olduğu dönemi yakalamaktır.

### Spektral indeksler
İndeksler iki veya daha fazla bandı birleştirerek belirli bir yüzey tipini öne çıkarır. Değerleri genelde −1 ile +1 arasındadır.

| İndeks | Formül | Ne gösterir | Yüksek değer |
|---|---|---|---|
| NDVI | (NIR − Kırmızı) / (NIR + Kırmızı) | Bitki örtüsü yoğunluğu | Orman, sulu tarım |
| NDBI | (SWIR1 − NIR) / (SWIR1 + NIR) | Yapılaşma | Beton, çatı, çıplak toprak |
| MNDWI | (Yeşil − SWIR1) / (Yeşil + SWIR1) | Su yüzeyi | Göl, deniz, baraj (> 0 genelde su) |
| NDMI | (NIR − SWIR1) / (NIR + SWIR1) | Bitki/yüzey nemi | Nemli bitki örtüsü |
| SAVI | 1.5 × (NIR − Kırmızı) / (NIR + Kırmızı + 0.5) | Toprak etkisi azaltılmış NDVI | Seyrek bitkili alanlarda NDVI'den daha güvenilir |
| BSI | ((SWIR1 + Kırmızı) − (NIR + Mavi)) / ((SWIR1 + Kırmızı) + (NIR + Mavi)) | Çıplak toprak | Çıplak arazi, nadas tarlası |

NDMI'yi ayrıca 1 km grid analizinde **nem göstergesi** olarak kullanacağız. ERA5-Land toprak nemi ~11 km çözünürlüklü olduğu için o ölçekte anlamlı değildir. Bu karar sınırlılıklarda da yazıyor.

## 2. Kompozit özeti: her şehir ve yıl için kaç görüntü var?
Kompozitin kalitesi, o yaz kaç bulutsuz görüntü bulunduğuna bağlıdır. 2014'te yalnızca Landsat 8 vardı. 2021'de Landsat 9 Ekim'de başladığı için yaz görüntüleri yine yalnızca Landsat 8'den gelir. 2024'te iki uydu birlikte olduğundan görüntü sayısı yaklaşık iki katına çıkmalı.

In [ ]:
rows = []
for city in CITY_LIST:
    for y in sorted(set(config.CHANGE_YEARS + [config.LABEL_YEAR])):
        n = gu.landsat_collection(geoms[city], y).size().getInfo()
        rows.append({"city": city, "year": y, "n_scenes": n})
scene_df = pd.DataFrame(rows).pivot(index="city", columns="year", values="n_scenes")
scene_df.to_csv(f"{config.TABLES_DIR}/kompozit_goruntu_sayilari.csv", encoding="utf-8-sig")
scene_df

## 3. Küçük test: İstanbul'da birkaç örnek nokta
Büyük hesaplamaları başlatmadan önce kodun doğru çalıştığını küçük bir örnekle görüyoruz. Sınıf başına 3 nokta alınıyor ve sonuç hemen geliyor.

**Etiket:** Her pikselin sınıfını ESA WorldCover 2021'den alıyoruz. WorldCover'ın 11 sınıfı `config.py`'daki tabloyla bizim 6 sınıfa indiriliyor. Ayrıca **homojenlik filtresi** uyguluyoruz: bir pikselin 3×3 komşuluğunda tek bir WorldCover sınıfı varsa o pikseli örnekliyoruz. Böylece "yarısı yol, yarısı tarla" gibi karışık kenar piksellerinden gelen etiket hatası azalıyor. Bunun bir bedeli var: model daha "kolay" piksellerle test edilir ve gerçek haritadaki doğruluk test sonucundan biraz düşük olabilir. Bu da bir sınırlılık maddesi.

**Kontrol edilecekler:** Yansıma bantları (SR_*) genelde 0 ile 0.6 arasında, indeksler −1 ile +1 arasında olmalı. `class` değerleri 0–5 olmalı.

In [ ]:
g = geoms["Istanbul"]
test_img = gu.summer_composite(g, config.LABEL_YEAR).select(config.FEATURES).addBands(gu.label_image())
test_fc = test_img.stratifiedSample(numPoints=3, classBand="class", region=g, scale=30,
                                    seed=config.RANDOM_STATE, tileScale=4)
test_df = pd.DataFrame([f["properties"] for f in test_fc.getInfo()["features"]])
test_df["class_name"] = test_df["class"].map(config.CLASSES)
display(test_df.round(3))
print(test_df[config.FEATURES].describe().loc[["min", "max"]].round(3))

## 4. Şehirlerin gerçek renkli görüntüleri (2014 ve 2024)
Kırmızı, yeşil ve mavi bantları birleştirerek gözün gördüğüne yakın bir görüntü üretiyoruz. Bunlar Faz 2'de veri tanıtımında ve raporda kullanılacak. Görüntüler küçük boyutlu (1024 piksel) indirildiği için hesaplama hafif.

In [ ]:
import requests
rgb_dir = f"{config.FIGURES_DIR}/rgb"
os.makedirs(rgb_dir, exist_ok=True)
for city in CITY_LIST:
    for y in config.CHANGE_YEARS:
        path = f"{rgb_dir}/{city}_{y}.png"
        if os.path.exists(path) and not FORCE_RECOMPUTE:
            continue
        g = geoms[city]
        vis = (gu.summer_composite(g, y).select(["SR_B4", "SR_B3", "SR_B2"])
               .visualize(min=0, max=0.3, gamma=1.2).clip(g))
        url = vis.getThumbURL({"region": g.bounds(), "dimensions": 1024, "format": "png"})
        with open(path, "wb") as f:
            f.write(requests.get(url, timeout=600).content)
        print("kaydedildi:", path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
fig, axes = plt.subplots(len(config.CHANGE_YEARS), len(CITY_LIST), figsize=(3 * len(CITY_LIST), 6))
for j, city in enumerate(CITY_LIST):
    for i, y in enumerate(config.CHANGE_YEARS):
        axes[i, j].imshow(mpimg.imread(f"{rgb_dir}/{city}_{y}.png"))
        axes[i, j].set_title(f"{city} {y}", fontsize=9)
        axes[i, j].axis("off")
plt.tight_layout()
plt.show()

## 5. Büyük hesaplamalar: il il Drive'a export
Aşağıdaki hesaplar 7 ilin tamamı üzerinde 30 m çözünürlükte yapılıyor. Konya tek başına ~40.000 km². Colab'da doğrudan beklenirse zaman aşımına uğrarlar. Bu yüzden Earth Engine'e **görev (task)** olarak gönderiyoruz. Görev Google'ın sunucusunda arka planda çalışır ve bitince CSV'yi Drive'daki `yapay-zeka-proje-gee` klasörüne yazar.

**Kota kurallarımız:**
- **İl il çalıştırma:** Her il ayrı bir görev ve ayrı bir dosya. Bir il hata verirse yalnızca o il yeniden çalıştırılır.
- **Önce pilot il:** Artvin önce tek başına çalışır ve harcadığı kota (EECU-saat) gösterilir. Diğer iller ondan sonra başlar.
- **Birleştirilmiş hesap:** Aynı ilin aynı yılı için kompozit bir kez kurulur. İndeksler, sıcaklık, su ve WorldCover oranları **aynı görevde** hesaplanır.
- **Önbellek:** `data/raw/gee/` içinde dosyası olan görev bir daha çalıştırılmaz.

**EECU-saat nedir?** Earth Engine'in hesaplama birimidir: kabaca "bir işlemci çekirdeğinin bir saatlik işi". Community tier'da aylık 150 EECU-saat hakkımız var. Her görevin harcadığı miktar `results/tables/gee_kota_kullanimi.csv` dosyasına kaydedilir.

Görevleri https://code.earthengine.google.com/tasks adresinden de izleyebilirsiniz. Colab bağlantısı koparsa görevler yine de çalışmaya devam eder: notebook'u tekrar çalıştırdığınızda biten dosyalar Drive'dan alınır.

### 5a. Örnek noktalar (2021, etiketli)
**Sınıf dengeli (stratified) örnekleme:** Her sınıftan eşit sayıda nokta alıyoruz. Rastgele alsaydık Konya'da noktaların çoğu tarım ve çayır olurdu, model az görülen sınıfları (ör. su) öğrenemezdi.

**Komşu piksel sızıntısı:** Yan yana iki piksel neredeyse aynı değerlere sahiptir. Biri eğitim, diğeri test kümesine düşerse model test sorusunun cevabını "ezberden" bilir ve başarı olduğundan yüksek görünür. Buna **veri sızıntısı** denir. Bunu azaltmak için noktalar arasında en az 300 m mesafe bırakıyoruz. Önce sınıf başına gerekenin 2 katı nokta alıyor, sonra mesafe filtresini Python'da uyguluyoruz (bölüm 6).

In [ ]:
RAW_GEE = f"{config.DATA_RAW}/gee"
os.makedirs(RAW_GEE, exist_ok=True)

n_classes = len(config.CLASSES)
per_class_raw = int(np.ceil(config.N_SAMPLES_PER_CITY / n_classes * config.OVERSAMPLE_FACTOR))
labels = gu.label_image()
SAMPLE_COLS = ["city", "class", "wc_code"] + config.FEATURES + ["lon", "lat"]

def city_samples(city):
    g = geoms[city]
    img = gu.summer_composite(g, config.LABEL_YEAR).select(config.FEATURES).addBands(labels)
    fc = img.stratifiedSample(numPoints=per_class_raw, classBand="class", region=g, scale=30,
                              seed=config.RANDOM_STATE, geometries=True, tileScale=8)
    return fc.map(with_lonlat).map(set_props({"city": city}))

print("Şehir başına sınıf başına ham nokta:", per_class_raw)

### 5b. 1 km grid: indeksler, sıcaklık, su ve WorldCover oranları (il başına tek görev)
Her ili 1 km × 1 km hücrelere bölüyoruz. Her hücre için içindeki 30 m piksellerin (~1.100 piksel) ortalamasını alıyoruz. Faz 4'te "bir hücrede yapılaşma arttıysa sıcaklık da arttı mı?" sorusunu bu hücreler üzerinden soracağız.

Grid, **eşit alanlı** bir projeksiyonda (EPSG:3035) kuruluyor. Böylece her hücre gerçekten 1 km² olur ve alanlar hücre oranlarının toplanmasıyla hesaplanabilir. 2014 ve 2024 aynı hücrelere düştüğü için iki yılı satır satır karşılaştırabiliyoruz.

| Sütun | Anlamı |
|---|---|
| `NDVI_2014` … `BSI_2024` | Hücredeki ortalama indeks |
| `LST_2014`, `LST_2024` | Landsat yüzey sıcaklığı ortalaması (°C) |
| `st_valid_YYYY` | Hücrede sıcaklık verisi olan piksel oranı (0–1) |
| `modis_lst_day_YYYY` | MODIS yaz gündüz yüzey sıcaklığı (°C) |
| `mndwi_water_YYYY` | Hücrenin MNDWI > 0 (su) olan oranı, 2014/2021/2024 |
| `jrc_water_2014`, `jrc_water_2021` | JRC'ye göre su oranı |
| `water_inter_2021`, `water_union_2021` | 2021'de JRC ve MNDWI'nin ikisinin de / en az birinin su dediği oran |
| `wc_10` … `wc_100` | Hücrenin WorldCover 2021 sınıf oranları |
| `elevation` | Ortalama rakım (SRTM, m) |

**Su alanı neden iki yöntemle?** JRC su verisi 2021'de bitiyor. 2024 su alanını bu yüzden Landsat'tan kendimiz hesaplıyoruz (**MNDWI > 0**). Bu yöntemin güvenilir olup olmadığını, iki verinin de bulunduğu 2021'de JRC ile karşılaştırarak kontrol ediyoruz (bölüm 8). JRC'de `waterClass` 2 (mevsimsel su) veya 3 (kalıcı su) olan pikseller su sayılıyor.

**WorldCover oranları neden burada?** Kar-buz, mangrov ve yosun-liken sınıflarından örnek almıyoruz. Model 2014/2024 haritalarında bu pikselleri mecburen 6 sınıftan birine atayacak; yüksek dağlardaki kar büyük olasılıkla "Çıplak" olacak. Bu sınıfların her ildeki alan oranını aynı görevde hesaplıyoruz (bölüm 7). Ayrı bir görev açmıyoruz.

Arazi örtüsü **tahmin** oranları (modelin 2014/2024'te hücrenin yüzde kaçını orman, yapı vb. bulduğu) Faz 4'te eklenecek.

In [ ]:
grid_proj = ee.Projection(config.GRID_CRS).atScale(config.GRID_SIZE_M)
WC_CODES = list(config.WORLDCOVER_TO_CLASS)
PER_YEAR = config.INDICES + ["LST", "st_valid"]
WATER_YEARS = sorted(set(config.CHANGE_YEARS + [config.LABEL_YEAR]))

def jrc_water(y):
    img = ee.ImageCollection(C["jrc_water"]).filter(ee.Filter.eq("year", y)).first()
    return img.select("waterClass").gte(2).unmask(0)

def city_grid_image(city):
    """Bir il için tüm 30 m katmanları tek görüntüde toplar ve 1 km'ye indirir."""
    g = geoms[city]
    comps = {y: gu.summer_composite(g, y) for y in WATER_YEARS}   # her yıl bir kez
    layers = []
    for y in config.CHANGE_YEARS:
        c = comps[y]
        st_valid = c.select("LST").mask().unmask(0).rename("st_valid")
        layers.append(c.select(config.INDICES + ["LST"]).addBands(st_valid)
                      .rename([f"{b}_{y}" for b in PER_YEAR]))
    mndwi = {y: comps[y].select("MNDWI").gt(config.MNDWI_WATER_THRESHOLD).unmask(0) for y in WATER_YEARS}
    for y in WATER_YEARS:
        layers.append(mndwi[y].rename(f"mndwi_water_{y}"))
    j14, j21 = jrc_water(2014), jrc_water(2021)
    m21 = mndwi[2021]
    layers += [j14.rename("jrc_water_2014"), j21.rename("jrc_water_2021"),
               j21.And(m21).rename("water_inter_2021"), j21.Or(m21).rename("water_union_2021")]
    wc = gu.worldcover()
    layers += [wc.eq(c).rename(f"wc_{c}") for c in WC_CODES]
    layers.append(ee.Image(C["dem"]).select("elevation"))

    # 30 m'ye sabitle → 1 km hücre ortalaması (oran katmanları için ortalama = alan oranı)
    stack_1km = (ee.Image.cat(layers).toFloat()
                 .reproject(crs=config.GRID_CRS, scale=config.LANDSAT_SCALE_M)
                 .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=2048)
                 .reproject(grid_proj))
    modis = [ee.ImageCollection(C["modis_lst"]).filterDate(f"{y}-06-01", f"{y}-09-01")
             .select("LST_Day_1km").mean().multiply(0.02).subtract(273.15)
             .rename(f"modis_lst_day_{y}").reproject(grid_proj) for y in config.CHANGE_YEARS]
    return ee.Image.cat([stack_1km] + modis + [ee.Image.pixelCoordinates(grid_proj)])

def city_grid(city):
    fc = city_grid_image(city).sample(region=geoms[city], projection=grid_proj,
                                      geometries=True, dropNulls=False, tileScale=8)
    return fc.map(with_lonlat).map(set_props({"city": city}))

GRID_COLS = (["city", "x", "y", "lon", "lat", "elevation"]
             + [f"{b}_{y}" for y in config.CHANGE_YEARS for b in PER_YEAR]
             + [f"modis_lst_day_{y}" for y in config.CHANGE_YEARS]
             + [f"mndwi_water_{y}" for y in WATER_YEARS]
             + ["jrc_water_2014", "jrc_water_2021", "water_inter_2021", "water_union_2021"]
             + [f"wc_{c}" for c in WC_CODES])
print(len(GRID_COLS), "sütun")

### 5c. Şehir bazında yıllık iklim serileri (2003–2024)
Her yıl ve her şehir için il sınırı içindeki ortalama değerler:
- **MODIS LST** (gündüz ve gece, yaz ortalaması, °C). MODIS her gün görüntü alır, 1 km çözünürlüklüdür. Uzun zaman serisi ve trend analizi için Landsat'tan daha uygundur.
- **ERA5-Land**: yaz ve yıllık ortalama toprak nemi (0–7 cm, m³/m³), yaz ortalaması 2 m hava sıcaklığı (°C). ERA5-Land bir **yeniden analiz** ürünüdür: ölçümlerle beslenen bir iklim modelinin çıktısıdır, doğrudan uydu ölçümü değildir.
- **CHIRPS**: yıllık ve yaz toplam yağış (mm). Kuraklık yıllarını yorumlamak için kullanılır.

Bu hesap 1 km ölçekte ve hafif olduğu için 7 il tek görevde yapılıyor.

In [ ]:
def climate_image(y):
    s, e = f"{y}-06-01", f"{y}-09-01"
    ys, ye = f"{y}-01-01", f"{y + 1}-01-01"
    modis = (ee.ImageCollection(C["modis_lst"]).filterDate(s, e)
             .select(["LST_Day_1km", "LST_Night_1km"]).mean()
             .multiply(0.02).subtract(273.15).rename(["modis_lst_day", "modis_lst_night"]))
    era_s = ee.ImageCollection(C["era5_land"]).filterDate(s, e).select(config.ERA5_BANDS).mean()
    era_y = (ee.ImageCollection(C["era5_land"]).filterDate(ys, ye)
             .select("volumetric_soil_water_layer_1").mean().rename("era5_sm_annual"))
    chirps = ee.ImageCollection(C["chirps"]).select("precipitation")
    return ee.Image.cat([
        modis,
        era_s.select("volumetric_soil_water_layer_1").rename("era5_sm_summer"),
        era_s.select("temperature_2m").subtract(273.15).rename("era5_t2m_summer"),
        era_y,
        chirps.filterDate(ys, ye).sum().rename("chirps_precip_annual"),
        chirps.filterDate(s, e).sum().rename("chirps_precip_summer"),
    ])

CLIMATE_BANDS = ["modis_lst_day", "modis_lst_night", "era5_sm_summer", "era5_t2m_summer",
                 "era5_sm_annual", "chirps_precip_annual", "chirps_precip_summer"]
climate_fc = ee.FeatureCollection([
    climate_image(y).reduceRegions(collection=cities_fc, reducer=ee.Reducer.mean(),
                                   scale=1000, tileScale=4).map(set_props({"year": y}))
    for y in config.MODIS_YEARS
]).flatten()
CLIMATE_COLS = ["city", "year"] + CLIMATE_BANDS

### 5d. Pilot: önce Artvin
Artvin için iki görev (örnek noktalar + grid) başlatılıyor. Bitince tabloda her görevin harcadığı **EECU-saat** ve aylık kotanın yüzde kaçı olduğu görünür. Diğer 6 ilin kullanımı kabaca il alanıyla orantılı olur. Aşağıdaki hücre bu oranla tüm iller için bir **kaba tahmin** de yazdırır.

`eecu_saat` boş görünürse Earth Engine kullanımı henüz raporlamamıştır. Bu durumda Tasks sayfasında göreve tıklayınca kullanım görünür.

In [ ]:
pilot = config.PILOT_CITY
pilot_usage = gu.run_exports({
    f"samples_2021_{pilot}": (city_samples(pilot), SAMPLE_COLS),
    f"grid_1km_{pilot}": (city_grid(pilot), GRID_COLS),
}, RAW_GEE, force=FORCE_RECOMPUTE)

if pilot_usage is not None:
    display(pilot_usage)
    pilot_eecu = pilot_usage["eecu_saat"].sum()
    km2 = {c: geoms[c].area(1000).divide(1e6).getInfo() for c in CITY_LIST}
    est = pilot_eecu * sum(km2[c] for c in CITY_LIST if c != pilot) / km2[pilot]
    print(f"Artvin toplam: {pilot_eecu:.3f} EECU-saat "
          f"(aylık kotanın %{100 * pilot_eecu / config.GEE_MONTHLY_EECU_HOURS:.2f}'i)")
    print(f"Kalan 6 il için alanla orantılı KABA tahmin: {est:.2f} EECU-saat "
          f"(%{100 * est / config.GEE_MONTHLY_EECU_HOURS:.1f})")

### 5e. Kalan iller ve iklim serisi
Pilot sonucu makulse (kaba tahmin aylık kotanın küçük bir kısmıysa) bu hücre kalan 6 ilin görevlerini ve iklim serisi görevini başlatır. Daha önce biten iller atlanır. Bir il hata verirse diğerleri yine kaydedilir. Hücreyi tekrar çalıştırınca yalnızca eksik iller hesaplanır.

`STOP_AFTER_PILOT = True` yaparsanız bu adım atlanır: pilot sonucuna bakıp karar vermek isterseniz bunu kullanın.

In [ ]:
STOP_AFTER_PILOT = False

if not STOP_AFTER_PILOT:
    rest = {}
    for city in CITY_LIST:
        if city == config.PILOT_CITY:
            continue
        rest[f"samples_2021_{city}"] = (city_samples(city), SAMPLE_COLS)
        rest[f"grid_1km_{city}"] = (city_grid(city), GRID_COLS)
    rest["climate_city_series"] = (climate_fc, CLIMATE_COLS)
    usage = gu.run_exports(rest, RAW_GEE, force=FORCE_RECOMPUTE)
    if usage is not None:
        display(usage)
        print(f"Bu adımda toplam: {usage['eecu_saat'].sum():.3f} EECU-saat")

kota = pd.read_csv(f"{config.TABLES_DIR}/gee_kota_kullanimi.csv")
print(f"Şimdiye kadar kaydedilen toplam kullanım: {kota['eecu_saat'].sum():.3f} EECU-saat "
      f"/ aylık {config.GEE_MONTHLY_EECU_HOURS}")

## 6. Örnek noktaları temizle: mesafe filtresi ve sınıf dengesi
1. Her şehirde noktalar rastgele sırayla geziliyor. Daha önce seçilmiş bir noktaya 300 m'den yakın olan nokta atılıyor.
2. Kalan noktalardan her şehir-sınıf için en fazla `2000 / 6 ≈ 333` nokta rastgele seçiliyor.

Bir ilde bir sınıf çok nadirse (ör. bir şehirde çok az çıplak alan varsa) o sınıf 333'ten az kalabilir. Bu doğal bir durum ve aşağıdaki tabloda görünüyor.

In [ ]:
from pyproj import Transformer
from scipy.spatial import cKDTree

raw = pd.concat([pd.read_csv(f"{RAW_GEE}/samples_2021_{c}.csv") for c in CITY_LIST], ignore_index=True)
to_m = Transformer.from_crs("EPSG:4326", config.GRID_CRS, always_xy=True)

def thin_points(df, min_dist, seed):
    x, y = to_m.transform(df["lon"].values, df["lat"].values)
    xy = np.column_stack([x, y])
    tree = cKDTree(xy)
    blocked = np.zeros(len(df), dtype=bool)
    keep = []
    for i in np.random.default_rng(seed).permutation(len(df)):
        if blocked[i]:
            continue
        keep.append(i)
        blocked[tree.query_ball_point(xy[i], r=min_dist)] = True
    return df.iloc[sorted(keep)]

thinned = pd.concat([thin_points(d, config.MIN_POINT_DISTANCE_M, config.RANDOM_STATE)
                     for _, d in raw.groupby("city")])
target = config.N_SAMPLES_PER_CITY // n_classes
samples = pd.concat([d.sample(min(len(d), target), random_state=config.RANDOM_STATE)
                     for _, d in thinned.groupby(["city", "class"])])
samples.insert(2, "class_name", samples["class"].map(config.CLASSES))
samples = samples[["city", "class", "class_name", "wc_code"] + config.FEATURES + ["lon", "lat"]]
samples = samples.reset_index(drop=True)
samples.insert(0, "sample_id", samples.index)
samples.to_csv(f"{config.DATA_PROCESSED}/samples_2021.csv", index=False)

counts = pd.concat({
    "ham": raw.groupby(["city", "class"]).size(),
    "mesafe_filtresi_sonrasi": thinned.groupby(["city", "class"]).size(),
    "nihai": samples.groupby(["city", "class"]).size(),
}, axis=1).fillna(0).astype(int).reset_index()
counts.insert(2, "class_name", counts["class"].map(config.CLASSES))
counts.to_csv(f"{config.TABLES_DIR}/ornek_sayilari.csv", index=False, encoding="utf-8-sig")

print("Nihai örnek sayısı:", len(samples))
low = counts[counts["nihai"] < target]
if len(low):
    print(f"\nHedefin ({target}) altında kalan şehir-sınıf çiftleri:")
    display(low)
samples.pivot_table(index="city", columns="class_name", values="lon", aggfunc="count", fill_value=0)

## 7. Dışarıda bırakılan WorldCover sınıflarının alan oranı
1 km hücrelerdeki WorldCover oranları toplanarak her ilin sınıf alanları (km²) bulunuyor. Merkezi il sınırı içinde kalan hücreler sayıldığı için toplam alan resmi il alanından biraz farklı olabilir. Oranlar bundan pek etkilenmez.

Bu tablo raporun **Sınırlılıklar** bölümüne gidecek. Oran düşükse (ör. %1'in altında) bu sınıfların yanlış atanması harita sonuçlarını pek etkilemez. Oran yüksekse (yüksek dağlık illerde kar gibi) ilgili sınıfın alan değişimi dikkatle yorumlanmalı.

In [ ]:
grid = pd.concat([pd.read_csv(f"{RAW_GEE}/grid_1km_{c}.csv") for c in CITY_LIST], ignore_index=True)
CELL_KM2 = (config.GRID_SIZE_M / 1000) ** 2
code_cols = [f"wc_{c}" for c in WC_CODES]
wc_areas = grid.groupby("city")[code_cols].sum() * CELL_KM2
wc_areas["toplam_km2"] = wc_areas[code_cols].sum(axis=1)

# Tüm WorldCover sınıflarının alan yüzdesi (Faz 2'de de kullanılacak)
wc_pct = pd.DataFrame({config.WORLDCOVER_NAMES[c]: 100 * wc_areas[f"wc_{c}"] / wc_areas["toplam_km2"]
                       for c in WC_CODES}).round(2)
wc_pct.to_csv(f"{config.TABLES_DIR}/worldcover_sinif_alanlari_yuzde.csv", encoding="utf-8-sig")

excluded = [c for c, v in config.WORLDCOVER_TO_CLASS.items() if v is None]
excl = wc_areas[["toplam_km2"]].copy()
for c in excluded:
    name = config.WORLDCOVER_NAMES[c]
    excl[f"{name}_km2"] = wc_areas[f"wc_{c}"]
    excl[f"{name}_yuzde"] = 100 * wc_areas[f"wc_{c}"] / wc_areas["toplam_km2"]
excl["dislanan_toplam_yuzde"] = excl[[col for col in excl if col.endswith("_yuzde")]].sum(axis=1)
excl = excl.round(3).reset_index()
excl.to_csv(f"{config.TABLES_DIR}/worldcover_dislanan_siniflar.csv", index=False, encoding="utf-8-sig")
excl

## 8. Su alanı karşılaştırması (JRC vs MNDWI)
- `mndwi_jrc_fark_yuzde_2021`: MNDWI'nin JRC'ye göre su alanını yüzde kaç fazla (+) veya eksik (−) bulduğu.
- `iou_2021` (Intersection over Union, kesişim/birleşim): İki yöntemin "su" dediği alanların kesişimi, birleşimine bölünür. 1 = tam uyum, 0 = hiç uyum yok. Kabaca 0.7 üstü iyi uyum sayılır.

Kıyı illerinde (İstanbul, İzmir, Samsun) kıyıdaki hücrelerin bir kısmı denize taşar. İki yöntem denizi farklı ölçüde "su" sayabileceği için bu illerde uyum biraz düşük çıkabilir.

Uyum iyiyse, 2024 su alanı için MNDWI'ye güvenebiliriz. Değişim analizinde tutarlılık için 2014 ve 2024'ün **ikisinde de** MNDWI kullanılacak, JRC ise yalnızca doğrulama için.

In [ ]:
water_cols = [f"mndwi_water_{y}" for y in WATER_YEARS] + ["jrc_water_2014", "jrc_water_2021",
                                                          "water_inter_2021", "water_union_2021"]
water = (grid.groupby("city")[water_cols].sum() * CELL_KM2)
water.columns = [c.replace("water_", "") + "_km2" for c in water.columns]
water["iou_2021"] = water["inter_2021_km2"] / water["union_2021_km2"]
water["mndwi_jrc_fark_yuzde_2021"] = 100 * (water["mndwi_2021_km2"] - water["jrc_2021_km2"]) / water["jrc_2021_km2"]
water["mndwi_degisim_yuzde_2014_2024"] = 100 * (water["mndwi_2024_km2"] - water["mndwi_2014_km2"]) / water["mndwi_2014_km2"]
water = water.round(3).reset_index()
water.to_csv(f"{config.TABLES_DIR}/su_alani_karsilastirma.csv", index=False, encoding="utf-8-sig")
water

## 9. İklim serisi ve 1 km grid: kaydetme ve Landsat LST eksik oranı
Landsat yüzey sıcaklığı (ST_B10) bazı piksellerde üretilmez. Bulut, gölge ya da algoritmanın gerekli girdiyi bulamaması bunun nedeni olabilir. Aşağıdaki tablo, her şehir ve yıl için kompozitte sıcaklık verisi **olmayan** piksellerin oranını veriyor (`st_valid` sütununun tersi). Ayrıca hiç sıcaklık değeri olmayan 1 km hücrelerin oranını da gösteriyor. Bu oranlar sınırlılıklarda raporlanacak.

In [ ]:
climate = (pd.read_csv(f"{RAW_GEE}/climate_city_series.csv")
             .sort_values(["city", "year"]).reset_index(drop=True))
climate.to_csv(f"{config.DATA_PROCESSED}/iklim_sehir_yillik.csv", index=False)
grid.to_csv(f"{config.DATA_PROCESSED}/grid_1km_2014_2024.csv", index=False)

rows = []
for city, d in grid.groupby("city"):
    for y in config.CHANGE_YEARS:
        rows.append({
            "city": city, "year": y, "hucre_sayisi": len(d),
            "lst_eksik_piksel_yuzde": 100 * (1 - d[f"st_valid_{y}"].mean()),
            "lst_tamamen_bos_hucre_yuzde": 100 * d[f"LST_{y}"].isna().mean(),
        })
lst_missing = pd.DataFrame(rows).round(2)
lst_missing.to_csv(f"{config.TABLES_DIR}/lst_eksik_orani.csv", index=False, encoding="utf-8-sig")
lst_missing

In [ ]:
# Hızlı mantık kontrolü: şehirlerin 2024 yaz değerleri
climate[climate["year"] == 2024].round(3)

## 10. Çıktı dosyaları özeti

In [ ]:
outputs = [
    f"{config.DATA_PROCESSED}/samples_2021.csv",
    f"{config.DATA_PROCESSED}/iklim_sehir_yillik.csv",
    f"{config.DATA_PROCESSED}/grid_1km_2014_2024.csv",
    f"{config.TABLES_DIR}/kompozit_goruntu_sayilari.csv",
    f"{config.TABLES_DIR}/ornek_sayilari.csv",
    f"{config.TABLES_DIR}/worldcover_sinif_alanlari_yuzde.csv",
    f"{config.TABLES_DIR}/worldcover_dislanan_siniflar.csv",
    f"{config.TABLES_DIR}/su_alani_karsilastirma.csv",
    f"{config.TABLES_DIR}/lst_eksik_orani.csv",
    f"{config.TABLES_DIR}/gee_kota_kullanimi.csv",
]
for p in outputs:
    df = pd.read_csv(p)
    print(f"{p:60s} {df.shape[0]:>7} satır × {df.shape[1]:>3} sütun")

## Faz 1 bitti mi?
- [ ] `samples_2021.csv`: 7 şehir, şehir başına ~2.000 nokta, 12 özellik + sınıf + koordinat
- [ ] `iklim_sehir_yillik.csv`: 7 şehir × 22 yıl = 154 satır
- [ ] `grid_1km_2014_2024.csv`: her ilin 1 km hücreleri, 2014 ve 2024 sütunları
- [ ] Yukarıdaki özet hücresi tüm dosyaların satır/sütun sayısını yazdırıyor

## Bu fazda ne yaptık, neden yaptık
- Her şehir ve yıl için bulutsuz bir yaz görüntüsü (medyan kompozit) ürettik. Yılları adil karşılaştırmak için hep aynı mevsimi kullandık.
- 6 Landsat bandına 6 spektral indeks ekledik. İndeksler bitki, su, yapı ve çıplak toprağı modelin daha kolay ayırmasını sağlar.
- WorldCover 2021'i etiket olarak kullanıp her şehirden sınıf dengeli, birbirinden en az 300 m uzak noktalar aldık. Amaç, modelin nadir sınıfları da öğrenmesi ve veri sızıntısının azalması.
- İklim verilerini (MODIS LST, ERA5-Land, CHIRPS) şehir ölçeğinde 22 yıllık seri olarak, Landsat sıcaklığı ve indeksleri de 1 km grid'de 2014 ve 2024 için çıkardık.
- Veri kısıtlarını sayıyla belgeledik: dışarıda bırakılan sınıfların oranı, JRC–MNDWI uyumu ve LST eksik oranı. Bunlar raporun Sınırlılıklar bölümüne gidecek.